# 1. Въведение (Introduction)

**Цел:**
Да се опише какво представлява Bonn EEG Dataset и защо се извършва изследователски анализ върху суровите данни.

**Какво се обяснява:**
*   че данните са сурови EEG времеви редове
*   че анализът е преди каквато и да е обработка или обучение
*   че целта е да се разбере естеството на сигналите и разликите между класовете

**Защо е важно:**
Показва, че проектът започва с разбиране на данните, а не директно с обучение на модели.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch

# Add src to path to import loader
sys.path.append(os.path.abspath('..'))
from src.data_processing.loader import load_bonn_data

# Configure plotting
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# 2. Описание на набора от данни (Dataset Overview)

**Цел:**
Да се представи структурата и основните характеристики на данните.

**Какво се разглежда:**
*   брой сигнали в набора
*   дължина на всеки EEG сигнал (брой времеви стъпки)
*   класовете и тяхното разпределение (seizure vs non-seizure)

**Защо е важно:**
*   доказва, че данните са подходящи за машинно и дълбоко обучение
*   показва дали има дисбаланс между класовете
*   аргументира избора на бинарна класификация

In [ ]:
# Define path to data
data_path = os.path.join(os.path.abspath('..'), 'data', 'raw')

# Load data
print(f"Loading data from: {data_path}")
X, y = load_bonn_data(data_path)

# Display basic information
n_samples, n_timesteps = X.shape
unique_classes, counts = np.unique(y, return_counts=True)

print(f"Total signals: {n_samples}")
print(f"Signal length: {n_timesteps} samples")
print(f"Classes: {unique_classes} (0=Non-Seizure, 1=Seizure)")
print(f"Class distribution: {dict(zip(unique_classes, counts))}")

# Visualize class distribution
plt.figure(figsize=(8, 5))
sns.barplot(x=['Non-Seizure (0)', 'Seizure (1)'], y=counts, palette='viridis')
plt.title('Class Distribution')
plt.ylabel('Number of Samples')
plt.show()

# 3. Визуален анализ на сурови EEG сигнали (Time-Domain Analysis)

**Цел:**
Да се визуализира как изглеждат суровите EEG сигнали във времето.

**Какво се разглежда:**
*   примерен non-seizure сигнал
*   примерен seizure сигнал
*   форма, амплитуда и вариабилност на сигналите

**Защо е важно:**
*   визуално се демонстрира разликата между нормална и епилептична мозъчна активност
*   дава интуитивно разбиране защо задачата е решима
*   осигурява фигури, които са много силни за дипломната работа

In [ ]:
# Select random samples from each class
idx_0 = np.where(y == 0)[0][0] # First Non-Seizure sample
idx_1 = np.where(y == 1)[0][0] # First Seizure sample

fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

# Plot Non-Seizure
axes[0].plot(X[idx_0], color='blue', alpha=0.7)
axes[0].set_title(f'Non-Seizure Signal (Class 0) - Sample {idx_0}')
axes[0].set_ylabel('Amplitude (µV)')

# Plot Seizure
axes[1].plot(X[idx_1], color='red', alpha=0.7)
axes[1].set_title(f'Seizure Signal (Class 1) - Sample {idx_1}')
axes[1].set_ylabel('Amplitude (µV)')
axes[1].set_xlabel('Time (samples)')

plt.tight_layout()
plt.show()

# 4. Статистически анализ на амплитудите

**Цел:**
Да се сравнят основни статистически характеристики на двата класа.

**Какво се анализира:**
*   средна стойност
*   стандартно отклонение
*   вариабилност на сигналите

**Защо е важно:**
*   показва количествени разлики между seizure и non-seizure
*   подкрепя твърдението, че епилептичните сигнали са по-хаотични
*   служи като аргумент за използване на по-сложни модели

In [ ]:
# Calculate basic statistics for each signal
means = np.mean(X, axis=1)
stds = np.std(X, axis=1)
mins = np.min(X, axis=1)
maxs = np.max(X, axis=1)

# Create DataFrame for easier plotting (optional, but explicit lists work too)
# We will use lists for simplicity
labels = ['Non-Seizure' if label == 0 else 'Seizure' for label in y]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Boxplot for Standard Deviation (Variability)
sns.boxplot(x=labels, y=stds, ax=axes[0], palette='Set2')
axes[0].set_title('Standard Deviation (Variability) Comparison')
axes[0].set_ylabel('Standard Deviation')

# Boxplot for Range (Max - Min)
ranges = maxs - mins
sns.boxplot(x=labels, y=ranges, ax=axes[1], palette='Set2')
axes[1].set_title('Signal Range (Max-Min) Comparison')
axes[1].set_ylabel('Range')

plt.tight_layout()
plt.show()

# 5. Анализ на разпределението на стойностите

**Цел:**
Да се изследва разпределението на амплитудите на EEG сигналите.

**Какво се разглежда:**
*   хистограми на амплитудите за двата класа
*   припокриване и различия между разпределенията

**Защо е важно:**
*   показва, че разпределенията не са идентични
*   демонстрира нелинейна структура на данните
*   аргументира използването на нелинейни модели (Random Forest, CNN)

In [ ]:
# Flatten arrays for histogram (take subset to avoid clutter)
X_0_flat = X[y==0].flatten()
X_1_flat = X[y==1].flatten()

# Randomly sample points to plot KDE without crashing memory
sample_size = 10000
X_0_sample = np.random.choice(X_0_flat, sample_size, replace=False)
X_1_sample = np.random.choice(X_1_flat, sample_size, replace=False)

plt.figure(figsize=(10, 6))
sns.kdeplot(X_0_sample, fill=True, label='Non-Seizure', color='blue')
sns.kdeplot(X_1_sample, fill=True, label='Seizure', color='red')

plt.title('Distribution of Signal Amplitudes (KDE)')
plt.xlabel('Amplitude')
plt.ylabel('Density')
plt.legend()
plt.show()

# 6. Честотен анализ на сигналите (Frequency-Domain Analysis)

**Цел:**
Да се анализира спектралното съдържание на EEG сигналите.

**Какво се разглежда:**
*   честотни компоненти на seizure и non-seizure сигнали
*   разлики в енергийните спектри

**Защо е важно:**
*   EEG сигналите носят ключова информация в честотната област
*   показва, че seizure сигналите имат различно спектрално поведение
*   оправдава използването на CNN, TCN и LSTM модели

In [ ]:
# Compute Power Spectral Density (PSD) using Welch's method
fs = 173.61  # Sampling frequency of Bonn Dataset

f_0, Pxx_0 = welch(X[y==0], fs=fs, nperseg=256, axis=1)
f_1, Pxx_1 = welch(X[y==1], fs=fs, nperseg=256, axis=1)

# Mean PSD across all samples for each class
mean_Pxx_0 = np.mean(Pxx_0, axis=0)
mean_Pxx_1 = np.mean(Pxx_1, axis=0)

plt.figure(figsize=(12, 6))
plt.semilogy(f_0, mean_Pxx_0, label='Non-Seizure', color='blue')
plt.semilogy(f_1, mean_Pxx_1, label='Seizure', color='red')

plt.title('Power Spectral Density (PSD)')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power (µV^2/Hz)')
plt.legend()
plt.grid(True)
plt.show()

# 7. Сравнителни наблюдения между класовете

**Цел:**
Да се обобщят визуалните и статистическите разлики между класовете.

**Какво се обсъжда:**
*   ключови различия във времевата и честотната област
*   кои характеристики най-силно разграничават класовете

**Защо е важно:**
*   подготвя почвата за feature engineering
*   подпомага избора на подходящи модели
*   прави преход към следващата фаза от проекта

Въз основа на извършения визуален и статистически анализ се открояват следните ключови зависимости:

*   **Времева област (Time-Domain):** Seizure сигналите (Клас 1) се отличават с многократно по-висока амплитуда и вариабилност. Средното стандартно отклонение при епилептичните пристъпи е значително по-голямо, което отразява хаотичната и интензивна електрическа активност на мозъка. Графиките на плътността (KDE) потвърждават, че разпределенията на стойностите за двата класа са коренно различни.
*   **Честотна област (Frequency-Domain):** Спектралният анализ (PSD) показва, че епилептичните сигнали съдържат значително повече енергия в целия честотен спектър. Тази ясна спектрална разлика е ключов индикатор, който валидира използването на честотни характеристики.

Тези наблюдения потвърждават, че избраната стратегия за бинарна класификация е добре обоснована, а данните съдържат достатъчно информативни характеристики за успешно обучение на невронни мрежи.

# 8. Ограничения на анализа

**Цел:**
Да се покаже критично мислене относно данните.

**Какво се отбелязва:**
*   анализът е върху кратки, предварително сегментирани сигнали
*   липсват мултиканални и дълги времеви записи
*   липсва пациент-специфична информация

**Защо е важно:**
*   демонстрира академична зрялост
*   предпазва от свръхобобщения
*   логично води към бъдеща работа с по-реалистични датасети

# 9. Изводи и следващи стъпки (Conclusions & Next Steps)

**Цел:**
Да се затвори EDA фазата и да се подготви преходът към моделиране.

**Какво се заключава:**
*   данните съдържат ясна дискриминативна информация
*   задачата е подходяща за ML и DL
*   следваща стъпка: обработка, сегментиране и моделиране

**Защо е важно:**
*   свързва анализа с останалата част от проекта
*   създава ясна и логична структура на дипломната работа